## TMDB Movie Data Analysis With Apache Spark

### STEP 1: Extract data from API

In [23]:
import requests
import os
import sys
from dotenv import load_dotenv
import datetime
import json

In [18]:
# load environment variables from .env file
load_dotenv()

# load environment variable
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

# set base url of TMDB website
TMDB_BASE_URL = 'https://api.themoviedb.org/3/movie'

In [21]:
import requests

def api_data_extraction(api_key, base_url, data_points):
    """
    This function extracts data from a URL using an API key.
    It returns a list of dictionaries, where each dictionary is the
    JSON data for a specific data point.
    
    Args:
        api_key (str): The API key for authentication.
        base_url (str): The base URL for the API endpoint (e.g., 'https://api.themoviedb.org/3/movie').
        data_points (list): A list of data point identifiers (e.g., movie IDs).

    Returns:
        list: A list of the successfully retrieved JSON data objects (dictionaries).
    """
    if not api_key:
        raise ValueError('API key is required!')

    results = []

    params = {
        'api_key': api_key,
        'append_to_response': 'credits'
    }

    for data_point in data_points:
        try:
            # Construct the full URL with the data point
            full_url = f'{base_url}/{data_point}'
            
            response = requests.get(full_url, params=params)
            
            response.raise_for_status()
            
            data = response.json()
            
            # Check for empty data, assuming an empty dictionary is an error
            if not data:
                print(f'Warning: Empty response for data point {data_point}. Skipping.')
                continue

            results.append(data)
            print(f'movie id {data_point} extracted successfully!')
        
        except requests.exceptions.HTTPError as http_error:
            # Log the specific error and continue to the next data point
            print(f'HTTP error for data point {data_point}: {http_error}')
            print(f'Response content: {http_error.response.text}')
            continue
        except requests.exceptions.Timeout as timeout_error:
            print(f"Timeout error for {data_point}: {timeout_error}")
            continue
        except requests.exceptions.ConnectionError as conn_error:
            print(f"Connection error for {data_point}: {conn_error}")
            continue
        except requests.exceptions.RequestException as req_error:
            print(f"An unexpected requests error occurred for {data_point}: {req_error}")
            continue
        except Exception as e:
            print(f"An unexpected error occurred for {data_point}: {e}")
            continue

    return results

In [14]:
# list of movies to extract
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 168259, 99861,
             284054, 12445, 181808, 330457, 351286, 109445, 321612, 260513]

In [22]:
# perform the data extraction
movie_data = api_data_extraction(TMDB_API_KEY, TMDB_BASE_URL, movie_ids)

HTTP error for data point 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?api_key=6383d74ea1616df62e4286bc3cb11c82&append_to_response=credits
Response content: {"success":false,"status_code":34,"status_message":"The resource you requested could not be found."}
movie id 299534 extracted successfully!
movie id 19995 extracted successfully!
movie id 140607 extracted successfully!
movie id 299536 extracted successfully!
movie id 597 extracted successfully!
movie id 135397 extracted successfully!
movie id 420818 extracted successfully!
movie id 24428 extracted successfully!
movie id 168259 extracted successfully!
movie id 99861 extracted successfully!
movie id 284054 extracted successfully!
movie id 12445 extracted successfully!
movie id 181808 extracted successfully!
movie id 330457 extracted successfully!
movie id 351286 extracted successfully!
movie id 109445 extracted successfully!
movie id 321612 extracted successfully!
movie id 260513 extracted successfull

In [33]:
# save the data as a raw file in data
def save_extracted_data(data, file_path):
    """
    Saves a list of data to a specified JSON file path.

    Args:
        data (list): A list of dictionaries to be saved.
        file_path (str): The full path to the output JSON file, including the filename.
    """

    directory = os.path.dirname(file_path)

    if directory and not os.path.exists(directory):
        try:
            os.makedirs(directory)
            print(f"Created directory: {directory}")
        except OSError as e:
            print(f"Error creating directory {directory}: {e}")
            return # Exit the function if directory creation fail

    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)
        print(f"Successfully saved data to {file_path}")
    except IOError as e:
        print(f'Error saving data to JSON file: {e}')
    except Exception as e:
        print(f"An unexpected error occurred while saving: {e}")


In [37]:

save_extracted_data(movie_data, 'data/movies_data_raw.json')

Error saving data to JSON file: [Errno 21] Is a directory: 'data/movies_data_raw.json'


In [38]:
len(movie_data)

18